# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
loaded = load_dotenv(".env")
if not loaded:
    loaded = load_dotenv("config.env")

print("📁 Working directory:", os.getcwd())

tavily_key = os.getenv("TAVILY_API_KEY")
print("TAVILY_API_KEY:", "OK ✅" if tavily_key else "MISSING ⚠️ (no es necesario para Part 01)")

# ❗ NO obligamos OPENAI_API_KEY porque vamos a usar embeddings locales (DefaultEmbeddingFunction)
openai_key = os.getenv("OPENAI_API_KEY")
print("OPENAI_API_KEY:", "OK ✅" if openai_key else "MISSING ⚠️ (no se usa con DefaultEmbeddingFunction)")

📁 Working directory: /workspace/Code/project/starter
TAVILY_API_KEY: OK ✅
OPENAI_API_KEY: OK ✅


### VectorDB Instance

### Collection

In [4]:
import chromadb

# Cliente persistente en disco (recomendado)
CHROMA_PATH = "chroma_udaplay_db"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

print("✅ chroma_client creado correctamente")

✅ chroma_client creado correctamente


In [5]:


chroma_client = chromadb.PersistentClient(path="chroma_udaplay_db_local")  # <-- NUEVA RUTA
embedding_fn = embedding_functions.DefaultEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name="udaplay_local",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

print("✅ BD y colección nuevas:", collection.name)

✅ BD y colección nuevas: udaplay_local


In [6]:
collection = chroma_client.get_or_create_collection(
    name="udaplay_local",
    embedding_function=embedding_fn
)


In [7]:
from chromadb.utils import embedding_functions

# Pick OpenAI Embedding Function (same one used for indexing & querying)
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="CHROMA_OPENAI_API_KEY",
    model_name="text-embedding-3-small"
)

In [8]:
import chromadb

# Cliente persistente (guarda la BD en disco)
CHROMA_PATH = "chroma_udaplay_db"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

print("✅ chroma_client creado (PersistentClient) en:", CHROMA_PATH)

✅ chroma_client creado (PersistentClient) en: chroma_udaplay_db


In [9]:
# TODO: Create a collection (recommended)
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}  # opcional, pero recomendado para embeddings OpenAI
)

print("✅ Colección lista:", collection.name)

✅ Colección lista: udaplay


In [10]:
import chromadb
from chromadb.utils import embedding_functions

# ✅ 1) Embeddings locales (no requieren API Key)
embedding_fn = embedding_functions.DefaultEmbeddingFunction()  # all-MiniLM-L6-v2 local [1](https://docs.trychroma.com/docs/embeddings/embedding-functions)

# ✅ 2) Cliente persistente
CHROMA_PATH = "chroma_udaplay_db_local"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# ✅ 3) Colección única y consistente
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}  # cosine distance (0 = más similar) [2](https://cookbook.chromadb.dev/core/collections/)
)

print("✅ Chroma listo:", CHROMA_PATH)

✅ Chroma listo: chroma_udaplay_db_local


In [11]:
# ✅ Ingest JSON files into Chroma collection (VectorDB)

data_dir = "games"
added = 0

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
    doc_id = os.path.splitext(file_name)[0]

    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

    added += 1

print(f"✅ Ingest completado. Archivos procesados: {added}")
print("✅ Total items en colección")

/home/student/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 80.3MiB/s]


✅ Ingest completado. Archivos procesados: 15
✅ Total items en colección


In [12]:
# ✅ DEMO: Semantic Search Query (RAG Retrieval Validation)

results = collection.query(
    query_texts=["juegos similares a Zelda"],
    n_results=3,
    include=["metadatas", "documents", "distances"]
)

print(results)

{'ids': [['009', '010', '008']], 'embeddings': None, 'documents': [["[Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", '[GameCube] Super Smash Bros. Melee (2001) - A crossover fighting game featuring characters from various Nintendo franchises battling it out in dynamic arenas.', '[Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'YearOfRelease': 1996, 'Publisher': 'Nintendo', 'Genre': 'Platformer', 'Name': 'Super Mario 64', 'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", 'Platform': 'Nintendo 64'}, {'YearOfRelease': 2001, 'Platform': 'GameCube', 'Publisher': 'Ni

## ADD DOCUMENTS:

In [13]:
# ✅ DEMO: Semantic Search (Pretty Print + relevance)

def pretty_print_query_results(query: str, n_results: int = 3):
    res = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["metadatas", "documents", "distances"]
    )

    metas = res["metadatas"][0]
    docs = res["documents"][0]
    dists = res["distances"][0]

    print(f"\nQuery: {query}")
    print("-" * 80)

    for i, (m, d, dist) in enumerate(zip(metas, docs, dists), start=1):
        print(f"[{i}] {m.get('Name')} | {m.get('Platform')} | {m.get('YearOfRelease')}")
        print(f"    Distance: {dist:.4f}  (lower = more similar)")
        print(f"    Description: {m.get('Description', d)[:220]}...")
        print()

pretty_print_query_results("juegos similares a Zelda", 3)

print("✅ Relevance note: The top results are considered relevant because their descriptions and themes "
      "are semantically close to the query intent (adventure/exploration/fantasy/puzzles).")


Query: juegos similares a Zelda
--------------------------------------------------------------------------------
[1] Super Mario 64 | Nintendo 64 | 1996
    Distance: 0.6724  (lower = more similar)
    Description: A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach....

[2] Super Smash Bros. Melee | GameCube | 2001
    Distance: 0.6810  (lower = more similar)
    Description: A crossover fighting game featuring characters from various Nintendo franchises battling it out in dynamic arenas....

[3] Super Mario World | Super Nintendo Entertainment System (SNES) | 1990
    Distance: 0.7005  (lower = more similar)
    Description: A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser....

✅ Relevance note: The top results are considered relevant because their descriptions and themes are semantically close to the query intent (adventure/exploration/fantasy/puzzles).
